In [11]:
import pandas as pd
df = pd.read_csv('data/books_dataset.csv')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13337 entries, 0 to 13336
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   work_id                    13337 non-null  int64  
 1   original_title             13337 non-null  object 
 2   author                     13337 non-null  object 
 3   original_publication_year  13337 non-null  int64  
 4   num_pages                  13337 non-null  int64  
 5   description                13337 non-null  object 
 6   image_url                  13337 non-null  object 
 7   reviews_count              13337 non-null  int64  
 8   text_reviews_count         13337 non-null  int64  
 9   5_star_ratings             13337 non-null  int64  
 10  ratings_count              13337 non-null  int64  
 11  avg_rating                 13337 non-null  float64
 12  genres_list                13337 non-null  object 
 13  similar_books_list         13337 non-null  obj

In [3]:
import pandas as pd
import re

def audit_descriptions(df: pd.DataFrame, col: str = "description") -> pd.DataFrame:
    """
    Audit the description column and flag potential issues.
    
    Returns a summary dataframe with per-entry flags.
    """
    html_pattern = re.compile(r"<[^>]+>")
    
    report = pd.DataFrame({
        "work_id"       : df["work_id"],
        "title"         : df["original_title"],
        "word_count"    : df[col].fillna("").apply(lambda x: len(x.split())),
        "has_html"      : df[col].fillna("").apply(lambda x: bool(html_pattern.search(x))),
        "is_empty"      : df[col].fillna("").apply(lambda x: x.strip() == ""),
        "is_short"      : df[col].fillna("").apply(lambda x: len(x.split()) < 10),
        "is_truncated_280"  : df[col].fillna("").apply(lambda x: len(x.split()) > 280),
        "is_truncated_370"  : df[col].fillna("").apply(lambda x: len(x.split()) > 370)
    })
    
    print("=== Description Audit ===")
    print(f"Total entries      : {len(report)}")
    print(f"Has HTML tags      : {report['has_html'].sum()}")
    print(f"Empty descriptions : {report['is_empty'].sum()}")
    print(f"Very short (<10w)  : {report['is_short'].sum()}")
    print(f"Will be truncated in bge-small-en-v1.5  : {report['is_truncated_370'].sum()}")
    print(f"Will be truncated in all-mpnet-base-v2  : {report['is_truncated_280'].sum()}")
    print(f"Avg word count     : {report['word_count'].mean():.0f}")
    print(f"Max word count     : {report['word_count'].max()}")
    
    return report

In [4]:
report = audit_descriptions(df)

=== Description Audit ===
Total entries      : 13337
Has HTML tags      : 0
Empty descriptions : 0
Very short (<10w)  : 4
Will be truncated in bge-small-en-v1.5  : 105
Will be truncated in all-mpnet-base-v2  : 544
Avg word count     : 155
Max word count     : 986


In [12]:
df['book_length'] = df['num_pages'].apply(
    lambda x: 'short' if x < 200 else 'medium' if x < 400 else 'long'
)
df[['num_pages', 'book_length']].head(10)

,num_pages,book_length
0,335,medium
1,335,medium
2,335,medium
3,335,medium
4,335,medium
5,335,medium
6,335,medium
7,335,medium
8,335,medium
9,335,medium


In [16]:
df[df['original_publication_year'] < 0]

,work_id,original_title,author,original_publication_year,num_pages,description,image_url,reviews_count,text_reviews_count,5_star_ratings,ratings_count,avg_rating,genres_list,similar_books_list,length_category,est_reading_hours,rating_category,publication_era,theme_label,book_length
368,1692879,Apologia,Plato,-390,127,The Apology of Socratesis Plato's version of t...,https://s.gr-assets.com/assets/nophoto/book/11...,47187,724,10191,23394,4.2,"['non-fiction', 'history', 'historical fiction...",[],Day Trip,2.1,Well Liked,Classics,Dystopian / Historical Fiction,short
480,288738,AEneis,Virgil,-19,442,The Aeneid (play /@'ni:Id/; Latin: Aeneis [aj'...,https://s.gr-assets.com/assets/nophoto/book/11...,144392,1903,25449,87273,3.8,"['poetry', 'fiction', 'history', 'historical f...",[],Epic Journey,7.4,Mixed Reviews,Classics,Dystopian / Historical Fiction,long
734,3098166,Oidipous Turannos,Sophocles,-430,75,"""...what man wins more happiness than just its...",https://s.gr-assets.com/assets/nophoto/book/11...,183832,2215,32497,137578,3.7,"['fiction', 'poetry', 'history', 'historical f...",[2115103],Day Trip,1.2,Mixed Reviews,Classics,Fantasy & Magic,short
10554,1052210,Antigone,Sophocles,-441,80,The curse placed on Oedipus lingers and haunts...,https://images.gr-assets.com/books/1486701308m...,112892,1975,17307,80377,3.6,"['fiction', 'poetry', 'history', 'historical f...",[],Day Trip,1.3,Mixed Reviews,Classics,Dystopian / Historical Fiction,short
11246,1842204,Bakkhai,Euripides,-405,96,Euripides' classic drama about the often morti...,https://images.gr-assets.com/books/1328704140m...,17191,374,2945,9967,3.8,"['fiction', 'poetry', 'history', 'historical f...",[],Day Trip,1.6,Mixed Reviews,Classics,Dystopian / Historical Fiction,short


In [ ]:
df = df[df['original_publication_year']  -500]

Removed rows with year -500. New shape: (13336, 20)


In [17]:
df['publication_era'].unique()

array(['Recent Releases', 'Late 20th Century', 'Older 20th Century',
       'Classics'], dtype=object)

In [18]:
df['length_category'].unique()

array(['Long Weekend', 'Day Trip', 'Epic Journey'], dtype=object)